In [ ]:
# Question 1 — Reading Payoff Matrices

# Payoff Matrix:
#           Left      Right
# Top     (3, 3)    (0, 5)
# Bottom  (5, 0)    (1, 1)

# (a) P1 plays Top, P2 plays Right → cell (Top, Right) = (0, 5)
#     Player 1 receives a payoff of 0.

# (b) Both players play Bottom → cell (Bottom, Bottom) — but P2 chooses columns.
#     Both play Bottom means P1=Bottom, P2=Right? No — "both play Bottom" is ambiguous
#     for a row/column game. Interpreting as P1=Bottom, P2=Right (the only 'Bottom' column
#     is not defined; the question likely means P1=Bottom, P2=Right since P2's strategies
#     are Left/Right, not Bottom). Re-reading: the question says 'both players play Bottom'
#     — since P2 has no 'Bottom', this likely means both choose their second strategy.
#     P1's second strategy = Bottom, P2's second strategy = Right → cell = (1, 1).
#     Player 2 receives a payoff of 1.

# (c) When P2 plays Left:  Top gives P1 payoff 3, Bottom gives P1 payoff 5.
#         → Bottom gives a higher payoff (5 > 3).
#     When P2 plays Right: Top gives P1 payoff 0, Bottom gives P1 payoff 1.
#         → Bottom gives a higher payoff (1 > 0).
#     In both cases, Bottom is the better strategy for Player 1.

In [ ]:
# Question 2 — Dominant Strategy Finder (2-Player)

def find_dominant_strategies(matrix):
    """
    Finds strictly dominant strategies for each player in a 2-player game.
    matrix: 2D list of (p1_payoff, p2_payoff) tuples.
    """
    num_rows = len(matrix)
    num_cols = len(matrix[0])

    # --- Player 1: check rows ---
    p1_dominant = None
    for r in range(num_rows):
        is_dominant = True
        for other_r in range(num_rows):
            if other_r == r:
                continue
            # Row r must beat other_r in every column
            for c in range(num_cols):
                if matrix[r][c][0] <= matrix[other_r][c][0]:
                    is_dominant = False
                    break
            if not is_dominant:
                break
        if is_dominant:
            p1_dominant = r
            break

    # --- Player 2: check columns ---
    p2_dominant = None
    for c in range(num_cols):
        is_dominant = True
        for other_c in range(num_cols):
            if other_c == c:
                continue
            # Column c must beat other_c in every row
            for r in range(num_rows):
                if matrix[r][c][1] <= matrix[r][other_c][1]:
                    is_dominant = False
                    break
            if not is_dominant:
                break
        if is_dominant:
            p2_dominant = c
            break

    # --- Output ---
    if p1_dominant is not None:
        print(f"Player 1 has a strictly dominant strategy: Row {p1_dominant}")
    else:
        print("Player 1 has no strictly dominant strategy.")

    if p2_dominant is not None:
        print(f"Player 2 has a strictly dominant strategy: Column {p2_dominant}")
    else:
        print("Player 2 has no strictly dominant strategy.")


# (i) Advertise / Not-Advertise
#           Advertise   Not-Advertise
# Advertise  (4, 4)       (6, 1)
# Not-Adv    (1, 6)       (3, 3)
print("=== Advertise / Not-Advertise ===")
advertise_matrix = [
    [(4, 4), (6, 1)],
    [(1, 6), (3, 3)]
]
find_dominant_strategies(advertise_matrix)

print()

# (ii) Prisoner's Dilemma
#            Cooperate   Defect
# Cooperate  (-1, -1)   (-3,  0)
# Defect     ( 0, -3)   (-2, -2)
print("=== Prisoner's Dilemma ===")
prisoners_dilemma = [
    [(-1, -1), (-3,  0)],
    [( 0, -3), (-2, -2)]
]
find_dominant_strategies(prisoners_dilemma)

print()

# (iii) Stag Hunt
#         Stag    Hare
# Stag   (4, 4)  (0, 3)
# Hare   (3, 0)  (3, 3)
print("=== Stag Hunt ===")
stag_hunt = [
    [(4, 4), (0, 3)],
    [(3, 0), (3, 3)]
]
find_dominant_strategies(stag_hunt)

In [ ]:
# Question 3 — IESDS Solver (2-Player)

def iesds(matrix, row_labels=None, col_labels=None):
    """
    Iterated Elimination of Strictly Dominated Strategies.
    matrix: 2D list of (p1_payoff, p2_payoff) tuples.
    row_labels / col_labels: optional names for rows and columns.
    """
    import copy

    num_rows = len(matrix)
    num_cols = len(matrix[0])

    # Track which rows/cols are still active
    active_rows = list(range(num_rows))
    active_cols = list(range(num_cols))

    if row_labels is None:
        row_labels = [str(i) for i in range(num_rows)]
    if col_labels is None:
        col_labels = [str(j) for j in range(num_cols)]

    iteration = 0

    while True:
        eliminated = False

        # Check each active row for P1
        for r in active_rows:
            for other_r in active_rows:
                if other_r == r:
                    continue
                # Is row r strictly dominated by other_r across all active columns?
                dominated = all(
                    matrix[r][c][0] < matrix[other_r][c][0]
                    for c in active_cols
                )
                if dominated:
                    print(f"Iteration {iteration+1}: Eliminating P1 row '{row_labels[r]}' "
                          f"(strictly dominated by row '{row_labels[other_r]}')")
                    active_rows.remove(r)
                    eliminated = True
                    iteration += 1
                    break
            if eliminated:
                break

        if eliminated:
            continue

        # Check each active column for P2
        for c in active_cols:
            for other_c in active_cols:
                if other_c == c:
                    continue
                dominated = all(
                    matrix[r][c][1] < matrix[r][other_c][1]
                    for r in active_rows
                )
                if dominated:
                    print(f"Iteration {iteration+1}: Eliminating P2 col '{col_labels[c]}' "
                          f"(strictly dominated by col '{col_labels[other_c]}')")
                    active_cols.remove(c)
                    eliminated = True
                    iteration += 1
                    break
            if eliminated:
                break

        if not eliminated:
            break

    print("\nSurviving strategies:")
    print(f"  P1 rows: {[row_labels[r] for r in active_rows]}")
    print(f"  P2 cols: {[col_labels[c] for c in active_cols]}")

    if len(active_rows) == 1 and len(active_cols) == 1:
        r, c = active_rows[0], active_cols[0]
        print(f"  → Unique predicted outcome: ({row_labels[r]}, {col_labels[c]}) "
              f"with payoffs {matrix[r][c]}")
    else:
        print("  → No unique outcome (multiple strategies survive).")


# 3x4 matrix from slides: P1 strategies A/B/C vs P2 strategies D/E/F/G
#          D        E        F        G
# A     (3,4)    (2,2)    (1,3)    (0,4)
# B     (4,3)    (3,4)    (2,2)    (1,1)
# C     (2,2)    (1,3)    (0,4)    (3,2)
print("=== 3x4 IESDS (A/B/C vs D/E/F/G) ===")
matrix_3x4 = [
    [(3,4), (2,2), (1,3), (0,4)],   # A
    [(4,3), (3,4), (2,2), (1,1)],   # B
    [(2,2), (1,3), (0,4), (3,2)],   # C
]
iesds(matrix_3x4, row_labels=['A','B','C'], col_labels=['D','E','F','G'])

In [ ]:
# Question 4 — Nash Equilibrium Finder: Pure Strategies (2-Player)

def find_pure_nash(matrix, row_labels=None, col_labels=None, game_name="Game"):
    """
    Finds all pure-strategy Nash Equilibria in a 2-player game.
    """
    num_rows = len(matrix)
    num_cols = len(matrix[0])

    if row_labels is None:
        row_labels = [f"R{i}" for i in range(num_rows)]
    if col_labels is None:
        col_labels = [f"C{j}" for j in range(num_cols)]

    # Best responses for P1: for each column, find row(s) maximising P1 payoff
    p1_br = {}  # col -> set of best-response rows
    for c in range(num_cols):
        best_payoff = max(matrix[r][c][0] for r in range(num_rows))
        p1_br[c] = {r for r in range(num_rows) if matrix[r][c][0] == best_payoff}

    # Best responses for P2: for each row, find col(s) maximising P2 payoff
    p2_br = {}  # row -> set of best-response cols
    for r in range(num_rows):
        best_payoff = max(matrix[r][c][1] for c in range(num_cols))
        p2_br[r] = {c for c in range(num_cols) if matrix[r][c][1] == best_payoff}

    # Nash Equilibria: cells that are BR for both
    nash_equilibria = []
    for r in range(num_rows):
        for c in range(num_cols):
            if r in p1_br[c] and c in p2_br[r]:
                nash_equilibria.append((r, c))

    print(f"=== {game_name} ===")
    if nash_equilibria:
        for (r, c) in nash_equilibria:
            print(f"  Nash Equilibrium: ({row_labels[r]}, {col_labels[c]}) "
                  f"→ payoffs {matrix[r][c]}")
    else:
        print("  No pure-strategy Nash Equilibrium found.")
    print()


# (i) Prisoner's Dilemma
pd = [[(-1,-1),(-3, 0)],[( 0,-3),(-2,-2)]]
find_pure_nash(pd, ['Cooperate','Defect'], ['Cooperate','Defect'], "Prisoner's Dilemma")

# (ii) Stag Hunt
sh = [[(4,4),(0,3)],[(3,0),(3,3)]]
find_pure_nash(sh, ['Stag','Hare'], ['Stag','Hare'], "Stag Hunt")

# (iii) Chicken
#           Swerve    Straight
# Swerve   (0, 0)    (-1, 1)
# Straight (1,-1)    (-10,-10)
chicken = [[(0,0),(-1,1)],[(1,-1),(-10,-10)]]
find_pure_nash(chicken, ['Swerve','Straight'], ['Swerve','Straight'], "Chicken")

# (iv) Battle of the Sexes
#          Opera   Football
# Opera   (2, 1)   (0, 0)
# Football(0, 0)   (1, 2)
bos = [[(2,1),(0,0)],[(0,0),(1,2)]]
find_pure_nash(bos, ['Opera','Football'], ['Opera','Football'], "Battle of the Sexes")

# (v) 3x3 matrix from slides (A/B/C vs X/Y/Z)
#        X       Y       Z
# A   (1,2)   (2,1)   (0,3)
# B   (3,0)   (1,2)   (2,1)
# C   (2,1)   (0,3)   (1,2)
matrix_3x3 = [
    [(1,2),(2,1),(0,3)],
    [(3,0),(1,2),(2,1)],
    [(2,1),(0,3),(1,2)],
]
find_pure_nash(matrix_3x3, ['A','B','C'], ['X','Y','Z'], "3x3 Slides Matrix")

In [ ]:
# Question 5 — Nash Equilibrium Finder: N-Player Game

import itertools

def find_nash_n_player(strategies, payoffs):
    """
    Finds all pure-strategy Nash Equilibria for an N-player game.
    strategies: list of lists — strategies[i] contains strategy names for player i.
    payoffs: dict mapping strategy-profile tuple -> list of payoffs (one per player).
    """
    n = len(strategies)
    all_profiles = list(itertools.product(*strategies))
    nash_equilibria = []

    for profile in all_profiles:
        is_nash = True
        for i in range(n):
            current_payoff = payoffs[profile][i]
            # Can player i profitably deviate?
            for alt_strategy in strategies[i]:
                if alt_strategy == profile[i]:
                    continue
                # Build the deviated profile
                deviated = list(profile)
                deviated[i] = alt_strategy
                deviated = tuple(deviated)
                if payoffs[deviated][i] > current_payoff:
                    is_nash = False
                    break
            if not is_nash:
                break
        if is_nash:
            nash_equilibria.append(profile)

    if nash_equilibria:
        print("Nash Equilibria found:")
        for eq in nash_equilibria:
            print(f"  Profile: {eq}  →  Payoffs: {payoffs[eq]}")
    else:
        print("No pure-strategy Nash Equilibrium exists.")

    return nash_equilibria


# 3-player Cooperate/Defect game
strategies = [['C','D'], ['C','D'], ['C','D']]
payoffs = {
    ('C','C','C'): [4, 4, 4],
    ('C','C','D'): [1, 1, 6],
    ('C','D','C'): [1, 6, 1],
    ('C','D','D'): [0, 3, 3],
    ('D','C','C'): [6, 1, 1],
    ('D','C','D'): [3, 0, 3],
    ('D','D','C'): [3, 3, 0],
    ('D','D','D'): [2, 2, 2],
}

print("=== 3-Player Cooperation Game ===")
find_nash_n_player(strategies, payoffs)

print()
print("Analysis:")
print("""
The only Nash Equilibrium is (D, D, D) with payoffs [2, 2, 2], even though
mutual cooperation (C, C, C) gives every player a higher payoff of 4.
This is unsurprising from a game-theory standpoint: it mirrors the classic
multi-player Prisoner's Dilemma, where Defect is a dominant strategy for each
individual regardless of what the others do.
The result illustrates the collective-action problem — rational self-interest
leads to a Pareto-inferior outcome, showing why voluntary cooperation on shared
projects often fails without binding agreements or enforcement mechanisms.
""")

In [ ]:
# Question 6 — Interactive Game Solver

def interactive_game_solver():
    """
    Lets the user input a 2-player payoff matrix at runtime and runs
    dominant strategy detection, IESDS, and Nash Equilibrium finding.
    """
    print("=" * 50)
    print("     Interactive Two-Player Game Solver")
    print("=" * 50)

    # Step (a): Get dimensions
    m = int(input("Enter the number of strategies for Player 1 (rows): "))
    n = int(input("Enter the number of strategies for Player 2 (columns): "))

    # Step (b): Get payoffs row by row
    matrix = []
    print("\nEnter payoffs row by row.")
    print("For each cell, type two space-separated integers: p1_payoff p2_payoff")
    for i in range(m):
        row = []
        for j in range(n):
            while True:
                try:
                    raw = input(f"  Row {i}, Col {j}: ")
                    p1, p2 = map(int, raw.split())
                    row.append((p1, p2))
                    break
                except ValueError:
                    print("  Invalid input. Please enter two integers separated by a space.")
        matrix.append(row)

    # Step (c): Display the matrix
    print("\n--- Payoff Matrix ---")
    col_header = "       " + "  ".join(f"Col{j}" for j in range(n))
    print(col_header)
    for i, row in enumerate(matrix):
        row_str = f"Row{i}:  " + "  ".join(str(cell) for cell in row)
        print(row_str)

    # Step (d): Run all three analyses
    print("\n" + "=" * 50)
    print("ANALYSIS 1: Dominant Strategies")
    print("=" * 50)
    find_dominant_strategies(matrix)

    print("\n" + "=" * 50)
    print("ANALYSIS 2: IESDS")
    print("=" * 50)
    iesds(matrix)

    print("\n" + "=" * 50)
    print("ANALYSIS 3: Pure-Strategy Nash Equilibria")
    print("=" * 50)
    find_pure_nash(matrix, game_name="User-Input Game")


# Run the interactive solver
# When prompted, enter the Battle of the Sexes matrix:
#   m=2, n=2
#   Row 0, Col 0: 2 1
#   Row 0, Col 1: 0 0
#   Row 1, Col 0: 0 0
#   Row 1, Col 1: 1 2
interactive_game_solver()